# Public Experiments

Public experiment configs are small package checks. They produce JSON metrics
that can be plotted later.

The residual reported by solver experiments is

$$
\|f_\theta(z)-z\|_2.
$$

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [
        Path.cwd(),
        Path("/content/silva-networks"),
        Path("/content/drive/MyDrive/silva-networks"),
    ]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import json
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

runner_path = root / "experiments/public/run_experiment.py"
spec = spec_from_file_location("public_runner", runner_path)
runner = module_from_spec(spec)
spec.loader.exec_module(runner)

Experiment configs are plain JSON. A solver config names the transition,
solver, damping, and iteration budget. The runner returns numerical quantities
that can be compared across configurations without changing the package code.

In [ ]:
config_path = root / "experiments/public/configs/solver_sweep.json"
config = json.loads(config_path.read_text())
metrics = runner.run_config(config)
metrics

Graph smoke experiments use the same loss as a normal PyTorch classifier:

$$
\mathcal L(\theta,\phi)
=
\operatorname{CE}(R_\phi(z^\star),y).
$$

The metrics dictionary records the loss curve and final accuracy.

In [ ]:
config_path = root / "experiments/public/configs/graph_silva_smoke.json"
config = json.loads(config_path.read_text())
metrics = runner.run_config(config)
metrics["losses"], metrics["accuracy"]

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})

plt.figure(figsize=(5, 3))
plt.plot(metrics["losses"], marker="o")
plt.xlabel("training step")
plt.ylabel("loss")
plt.tight_layout()

A fully configurable stack is still just JSON. The configuration below chooses
each local branch, global branch, self branch, and solver separately:

$$
z_\ell^\star=f_{\theta_\ell}(z_\ell^\star,h_{\ell-1}),
\qquad
h_\ell=z_\ell^\star.
$$

Layer $\ell$ receives its own $L_\ell$, $G_\ell$, $H_\ell$, and
`SolverConfig`.

In [ ]:
config_path = root / "experiments/public/configs/fully_configurable_graph.json"
config = json.loads(config_path.read_text())
config["device"] = "cpu"
metrics = runner.run_config(config)
metrics["output_shape"], metrics["state_shape"], metrics["solver_residuals"]

## Citation and Sources

If this notebook or package is used, cite the software repository:

```text
Dr. Jose Luis Silva. SILVA Networks. Version 1.0.0. MIT License.
https://github.com/jseluis/silva-networks
```

When the work is connected to the SILVA Networks paper, cite the paper as well:

```text
Jose Luis Lima de Jesus Silva. SILVA Networks as Structured Implicit Layers and
Vector Attractors via Dynamic Interaction Fields. 2026. arXiv:2607.28989.
https://arxiv.org/abs/2607.28989
```

Background references used in the tutorial suite include:

- Deep Equilibrium Models, Bai, Kolter, and Koltun, NeurIPS 2019:
  https://arxiv.org/abs/1909.01377
- Multiscale Deep Equilibrium Models, Bai, Koltun, and Kolter, NeurIPS 2020:
  https://arxiv.org/abs/2006.08656
- Stabilizing Equilibrium Models by Jacobian Regularization, Bai, Koltun, and
  Kolter, ICML 2021: https://arxiv.org/abs/2106.14342
- Graph Attention Networks, Velickovic et al., ICLR 2018:
  https://arxiv.org/abs/1710.10903
- Attention Is All You Need, Vaswani et al., 2017:
  https://arxiv.org/abs/1706.03762